# Julia notebook to compute the solution to the frictional geostrophic equations with an unstratified fluid, a flat bottom, and a specified pressure gradient or surface wind stress.

twnh Nov '25

This notebook solves

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the surface wind stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$, but no baroclinic pressure.

This code specifies one of (i) surface wind stress, or (ii) surface pressure gradient, and then solves the problem.

In [ ]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit
using Plots
using Statistics
notebook_name = "Example_barotropic_GeostrophicFlow_v0.3.ipynb"
using Infiltrator

"Example_barotropic_GeostrophicFlow_v0.3.ipynb"

### Define symbols and parameter values

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x)
geometry_params = (H(x,y), z, ξ)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀, ϕ  = symbols("ν₀ ϕ",  real=true, positive=true) # Viscosity parameters
τˣ, τʸ = symbols("τˣ τʸ", real=true)                # Surface wind stress components
τs     = symbols("τs",    complex=true)             # Complex surface wind stress
pg     = symbols("pg",    complex=true)             # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)

# Define symbolic viscosity here:
ν = ν₀                                              # Constant viscosity profile
uv_params = (f, ϵ, ν)

# Define the parameter values
f_val  = 1
ϵ_val  = 0.95
ν₀_val = 0.06
# ν₀_val = 0.001
display("Non-dimensional Ekman-layer depth:")
Ekman_depth = sqrt(2*ν₀_val/f_val)
display(Ekman_depth)

# Compute compound parameter:
ϕ_val = sqrt(f_val / ν₀_val) / ϵ_val

# Define dictionaries for substitutions:
param_values = Dict(f=>f_val, ϵ=>ϵ_val, ν₀=>ν₀_val, ϕ=>ϕ_val)

# Define specific case values here:
Hfn = 1.0

if true
    pg_val = 0.1 + 0.0im
    case_values  = Dict(pg=>pg_val, H(x, y)=>Hfn) 
else
    τs_val = 0.1 + 0.0im        # Similar to Vallis (2006) Fig. 2.12b
    case_values  = Dict(τs=>τs_val, H(x, y)=>Hfn) 
end

display("Case values:")
display(case_values)

### Function to solve the frictional geostrophic equation using a Green's function:

In [ ]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, ϵ, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                            # Unknown coefficient in the Green's function solution

    #0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ϵ^2 *  ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))


    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final simplify (to cancel constants). Avoid simplify in general because it's not always reproducible.
    G = simplify(G)
    return G
end ;

### Compute the G's function and various manipulations of it:

In [ ]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
display("Full Guv(x,ξ):")
display(Guv_sym)

Guv = Guv_sym.subs(f,ϕ^2 * ϵ^2 * ν₀)
display("Simplified Guv(z,ξ)")
display(Guv)

Guv0 = Guv.subs(ξ,0)
display("Simplified Guv(z,0)")
display(Guv0)

tmp  = diff(Guv, z)
tmp2 = tmp.subs(z, -H(x,y))
dGuv_dz_at_bottom = tmp2.args[1].args[1]
display("Simplified d/dz Guv(z,ξ) @ z = -H(x,y)")
display(dGuv_dz_at_bottom)
display("Simplified d/dz Guv(z,0) @ z = -H(x,y)")
display(dGuv_dz_at_bottom.subs(ξ,0))

tmp = integrate(expand(Guv), (z, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(ξ, -H(x, y))
Guv_int_wrt_z = tmp.subs(max_obj, ξ)
display("Simplified integral Guv(z,ξ) wrt z = -H(x,y) to 0")
display(factor(Guv_int_wrt_z))

Guv_int_wrt_z0 = Guv_int_wrt_z.subs(ξ,0)
display("Simplified integral Guv(z,ξ) wrt z = -H(x,y) to 0 @ ξ = 0")
display(simplify(Guv_int_wrt_z0))
tmp = integrate(expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)
display("Simplified integral Guv(z,ξ) wrt ξ = -H(x,y) to 0")
display(simplify(factor(Guv_int_wrt_ξ)))

𝔲1 = Guv_int_wrt_ξ * pg
display("Pressure-driven flow field 𝔲₁(x,y,z):")
display(simplify(𝔲1))

𝔲2 = - Guv0 * τs 
display("Stress-driven   flow field 𝔲₂(x,y,z):")
display(simplify(𝔲2))
𝔲 = 𝔲1 + 𝔲2

# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * f * 𝔲1 + ϵ^2 * diff(diff(ν * 𝔲1,z),z)
tmp1 = simplify(tmp1.subs(ϕ,sqrt(f/ν₀)/ϵ))
tmp2 = -im * f * 𝔲2 + ϵ^2 * diff(diff(ν * 𝔲2,z),z)
tmp2 = simplify(tmp2.subs(ϕ,sqrt(f/ν₀)/ϵ))
@assert tmp1 + tmp2 == pg

tmp = integrate(expand(Guv_int_wrt_ξ),(z,-H(x,y),0))
𝔘1 = (tmp.args[1] + tmp.args[2].args[1].args[1])*pg
display("Pressure-driven flow field 𝔘₁(x,y):")
display(simplify(𝔘1))

𝔘2 = - Guv_int_wrt_z0 * τs 
display("Stress-driven   flow field 𝔘₂(x,y):")
display(simplify(𝔘2))
𝔘 = 𝔘1 + 𝔘2
tmp    = integrate(𝔲,(z,-H(x, y),0))
int_𝔲  = tmp.args[1] + tmp.args[2].args[1].args[1]
res    = int_𝔲 - 𝔘
@assert res == 0

τb1 = - ϵ^2 * ν * (integrate(expand(dGuv_dz_at_bottom),(ξ,-H(x,y),0)).args[1].args[1]) * pg
display("Pressure-driven bottom stress on fluid -ν ϵ^2 d/dz u(x,y,z) @ z = -H:")
display(simplify(τb1))

dGuv_dz_at_bottom_and_top = dGuv_dz_at_bottom.subs(ξ,0)
τb2 =   ϵ^2 * ν * dGuv_dz_at_bottom_and_top * τs 
display("Stress-driven   bottom stress on fluid - ν ϵ^2 d/dz u(x,y,z) @ z = -H")
display(simplify(τb2))
τb = τb1 + τb2

# Check expression for bottom stress on fluid:
tmp = - ϵ^2 * ν * diff(𝔲,z).subs(z,-H(x,y))
@assert simplify(tmp - τb) == 0

# Check expression for surface stress on fluid:
@assert simplify(diff(𝔲1,z).subs(z,0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ϵ^2 * ν * diff(𝔲,z).subs(z,0)
@assert simplify(tmp - τs) == 0

### Form final equation for $\frak{U}$:

In [ ]:
lhs = im * f * 𝔘
rhs = - H(x,y)*pg - τb + τs
println()
display("Final equation linking surface pressure pₛ(x,y) to windstress:")
display(Eq(lhs, rhs))

### Solve for windstress given the pressure gradient, or vice versa:

In [ ]:
eqn = expand(lhs - rhs)
eqn = eqn.subs(case_values)
if haskey(case_values, pg)
    sol = solve(Eq(eqn,0), τs)[1]
    sol_num = float(numerator(  sol).subs(param_values).n())
    sol_den = float(denominator(sol).subs(param_values).n())
    τs_val = float(sol_num / sol_den)
    case_values[τs] = τs_val
    display("Computed surface wind stress τs:")
    display(τs_val)
else
    sol = solve(Eq(eqn,0), pg)[1]
    sol_num = float(numerator(  sol).subs(param_values).n())
    sol_den = float(denominator(sol).subs(param_values).n())
    pg_val = float(sol_num / sol_den)
    case_values[pg] = pg_val
    display("Computed surface pressure gradient pg:")
    display(pg_val)
end  
println()

### Solve for $\frak{u}$ and $\frak{U}$ and bottom stress:

In [ ]:
𝔘_val = 𝔘.subs(case_values)
𝔘_val = 𝔘_val.subs(param_values)
Uval  = float(real(𝔘_val))
Vval  = float(imag(𝔘_val))
display("Depth-integrated flow field 𝔘(x,y):")
display(float(𝔘_val.n()))

𝔲_val = 𝔲.subs(case_values)
𝔲_val = 𝔲_val.subs(param_values)
𝔲_fld(zz) = 𝔲_val.subs(z,zz).n()
N_pts = 512
zvals = range(-Hfn, 0, length=N_pts)
uvals = [𝔲_fld(z) for z in zvals]
sum_uvals = sum(uvals) * (Hfn / length(zvals))
display("Vertical sum of 𝔲(z) from z = -H to 0 with $N_pts evaluation points:")
display(sum_uvals)
println()

τb_val = τb.subs(case_values)
τb_val = τb_val.subs(param_values).n()
display("Bottom stress on fluid - ν ϵ^2 d/dz u(x,y,z) @ z = -H:")
display(τb_val)

# Check surface and bottom stress estimates:
duvals_dz = diff(uvals) * (N_pts / Hfn)
display("Numerical bottom stress on fluid estimate:")
display(-duvals_dz[1] * (ν₀_val * ϵ_val^2))
println()
display("Surface stress from symbolic solution ν ϵ^2 d/dz u(x,y,z) @ z = 0:")
display(τs_val)
display("Numerical surface stress on fluid estimate:")
display(duvals_dz[end] * (ν₀_val * ϵ_val^2))
println()

# Check force balance:
@assert abs(im * f_val * float(𝔘_val) + Hfn * pg_val + τb_val - τs_val) < 1e-10

### Plot vertical profiles of $u(z), v(z)$, $U, V$, and the vector force balance: 

In [ ]:
plt1 = plot()   # Start with empty plot
plot( real.(uvals), zvals, label="u(z)", lw=2, color=:blue, linestyle=:solid)
plot!(imag.(uvals), zvals, label="v(z)", lw=2, color=:red, linestyle=:solid)
plot!( [Uval, Uval], [-Hfn, 0], label="U", lw = 2, color=:blue, linestyle=:dash)
plot!( [Vval, Vval], [-Hfn, 0], label="V", lw = 2, color=:red, linestyle=:dash)
ylabel!("z")
xlabel!("Flow value")
# tit_str = "\$ \\nabla p \$ = ($(round(pg_val,digits=3)), 0), \$H_{Ek}\$=$(round(Ekman_depth,digits=3))"
tit_str = "\$H_{Ek}\$=$(round(Ekman_depth,digits=3))"
title!("Flow \$(u(z), v(z)), (U, V)\$: " * tit_str, titlefontsize=8)
display(plt1)
savefig(notebook_name*"_VerticalProfiles.pdf")

### Plot balance of terms in the final equation:

In [ ]:
term_names = ["Coriolis force", "Pressure gradient force", "Bottom Stress", "Wind Stress"]
terms = [-im * f_val * float(𝔘_val), -Hfn * pg_val, -τb_val, τs_val]
start = [0, terms[1], terms[1] + terms[2] , terms[1] + terms[2] + terms[3]]
colors = [:red, :blue, :green, :orange]   # Assign each a color
plt2 = plot()   # Start with empty plot
max_term = float(maximum(abs.(terms)))
axis_lims = 1.05 * max_term *[ -1; 1]
for i in eachindex(terms)
    quiver!(
        [real(start[i])], [imag(start[i])],
        quiver=([real(terms[i])], [imag(terms[i])]),
        arrow=true, aspect_ratio=:equal, color=colors[i], label=term_names[i], xlims=axis_lims, ylims=axis_lims
    )
    scatter!([real(start[i] + terms[i])], [imag(start[i] + terms[i])], color=colors[i], label=term_names[i], markerstrokewidth=0)
end

plot!(legend=:topright, xlabel="x-component", ylabel="y-component", aspect_ratio=:equal, title="Term Balance in Final Equation: " * tit_str, titlefontsize=8)
display(plt2)    # Show the plot
savefig(notebook_name*"_TermBalance.pdf")

### Plot hodograph

In [ ]:
plt3 = plot()   # Start with empty plot
plot( real.(uvals), imag.(uvals), label="(u(z),v(z))", lw=2, color=:blue, linestyle=:solid)
scatter!( [Uval], [Vval], label="(U,V)", color=:red, markerstrokewidth=0)
max_val = float(maximum(abs.([real.(uvals);imag.(uvals)])))
axis_lims = max_val*1.05*[-1; 1]
for i in N_pts:-64:1
    quiver!(
        [0.0], [0.0],
        quiver=([real(uvals[i])], [imag(uvals[i])]),
        arrow=true, aspect_ratio=:equal, xlims=axis_lims, ylims=axis_lims, color=:blue
    )
end
quiver!(
        [0.0], [0.0],
        quiver=([Uval], [Vval]),
        arrow=true, aspect_ratio=:equal, xlims=axis_lims, ylims=axis_lims, color=:red, linestyle=:dash, lw = 4
    )

plot!(legend=:bottomright, xlabel="x-component", ylabel="y-component", aspect_ratio=:equal, title="Hodograph: " * tit_str, titlefontsize=8)
display(plt3)    # Show the plot
savefig(notebook_name*"_Hodograph.pdf")